# Module 4: `conv.py` — 2D Convolution (`im2col`/`col2im`) & In-Silico Physics Benchmarks

This notebook demonstrates vectorized 2D convolution via `im2col`, overlapping patch gradient accumulation via `col2im` (`+=`), and our two In-Silico High Energy Physics benchmarks (4D Collider Kinematics `MLP` and $8\times 8$ Calorimeter Shower `Conv2d`).

## 1. Imports & Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import numpy as np
from tensorgrad.tensor import Tensor, numerical_grad_check
from tensorgrad.conv import Conv2d, im2col, col2im, run_collider_mlp_benchmark, run_calorimeter_cnn_benchmark
print('Loaded tensorgrad.conv successfully.')

Loaded tensorgrad.conv successfully.


## 2. `im2col` Receptive-Field Unrolling & `Conv2d` Gradient Check
For an $8\times 8$ calorimeter image and a $3\times 3$ kernel with `padding=0, stride=1`, `im2col` extracts $6\times 6 = 36$ windows of length $9$ (`shape = (36, 9)`).

In [2]:
sample_img = np.ones((1, 1, 8, 8), dtype=np.float64)
cols, h_out, w_out = im2col(sample_img, k_h=3, k_w=3, stride=1, padding=0)
print(f'Output spatial dims: ({h_out}, {w_out}) | im2col matrix shape: {cols.shape}')

rng = np.random.default_rng(31)
X_small = Tensor(rng.normal(size=(2, 2, 5, 5)))
conv_layer = Conv2d(in_channels=2, out_channels=3, kernel_size=3, stride=1, padding=1, rng=rng)
head_w = rng.normal(size=(2, 3, 5, 5))
chk_x = numerical_grad_check(lambda t: (conv_layer(t) * head_w).sum(), X_small)
print(f"Conv2d input X (col2im +=) gradcheck: passed={chk_x['passed']}, rel_error={chk_x['rel_error']:.3e}")

Output spatial dims: (6, 6) | im2col matrix shape: (36, 9)
Conv2d input X (col2im +=) gradcheck: passed=True, rel_error=1.103e-10


## 3. In-Silico Physics Benchmarks (Collider `MLP` & Calorimeter `Conv2d`)
We execute both physics classification benchmarks (guarded with lightweight synthetic sample sizes).

In [3]:
RUN_HEAVY = False
steps_mlp = 120 if not RUN_HEAVY else 300
res_mlp = run_collider_mlp_benchmark(seed=42, steps=steps_mlp)
res_cnn = run_calorimeter_cnn_benchmark(seed=7, steps=45)
print(f"[1] Collider 4D MLP  : Loss {res_mlp['initial_loss']:.4f} -> {res_mlp['final_loss']:.4f} | Accuracy = {res_mlp['accuracy']*100:.1f}%")
print(f"[2] Calorimeter Conv2d: Loss {res_cnn['initial_loss']:.4f} -> {res_cnn['final_loss']:.4f} | Accuracy = {res_cnn['accuracy']*100:.1f}%")

[1] Collider 4D MLP  : Loss 0.7266 -> 0.0015 | Accuracy = 100.0%
[2] Calorimeter Conv2d: Loss 0.5554 -> 0.0000 | Accuracy = 100.0%


## Conclusion
`conv.py` completes the `tensorgrad` engine by reducing 2D spatial convolutions to fast matrix multiplications (`im2col`) and exact adjoint patch accumulation (`col2im`), achieving 100% classification accuracy on both synthetic collider and calorimeter benchmarks.